In [1]:
import gymnasium as gym
from gymnasium.spaces import Discrete

import numpy as np
from tqdm.auto import tqdm

from dataclasses import dataclass

In [2]:
type State = int
type Action = int
type Reward = float

In [3]:
@dataclass
class Parameters:
  alpha: float  # step size
  epsilon: float  # e-greedy
  epsilon_decay: float
  epsilon_end: float
  gamma: float  # discount factor
  

def select_action(q_table: np.ndarray, state: State, env: gym.Env, epsilon: float) -> Action:
  # using e-greedy method
  
  assert isinstance(env.action_space, Discrete)
  n_actions = env.action_space.n
  
  q: np.ndarray = q_table[state]
  assert q.shape == (n_actions,)

  if env.np_random.random() < epsilon: # random
    return env.action_space.sample()
  
  return int(np.random.choice(np.flatnonzero(q == q.max())))

In [4]:
def update(
  q_table: np.ndarray,
  state: State,
  new_state: State,
  action: Action,
  new_action: Action,
  reward: Reward,
  done: bool,
  params: Parameters
):
  q_next = (1 - done)*q_table[new_state, new_action]
  td_error = reward + params.gamma*q_next - q_table[state, action]
  q_table[state, action] += params.alpha*td_error


In [5]:
environ: gym.Env[int, int] = gym.make("FrozenLake-v1", max_episode_steps=10_000, is_slippery=True)

In [6]:
assert isinstance(environ.action_space, Discrete)
assert isinstance(environ.observation_space, Discrete)


n_actions = int(environ.action_space.n)
obs_dim = int(environ.observation_space.n)

In [7]:
# init params
params = Parameters(alpha=0.1, epsilon=0.1, epsilon_decay=0.998, epsilon_end=0.01, gamma=0.99)
n_episodes = 10_000

In [8]:
# init q table
q_table = np.zeros((obs_dim, n_actions))

total_rewards = []

pbar_ep = tqdm(range(n_episodes), desc="Episodes")
for ep in pbar_ep:
  # reset env
  obs, info = environ.reset()
  done = False  
  total_reward = 0
  action = select_action(q_table, obs, environ, params.epsilon)

  while not done:
    # take the action
    new_obs, reward, terminated, truncated, info = environ.step(action)
    done = terminated or truncated

    new_action = select_action(q_table, new_obs, environ, params.epsilon)

    update(q_table, obs, new_obs, action, new_action, float(reward), done, params)

    action, obs = new_action, new_obs

    params.epsilon = max(params.epsilon_end, params.epsilon*params.epsilon_decay)

    total_reward += float(reward)
  
  total_rewards.append(total_reward)
  print(f"Episode finished! Total reward: {total_reward}")


Episodes:   0%|          | 0/10000 [00:00<?, ?it/s]

Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total reward: 0.0
Episode finished! Total rewa

In [9]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
  y=total_rewards,
  mode='lines',
  name='Raw',
  line=dict(color='steelblue')
))

fig.update_layout(
  title='Training Rewards',
  xaxis_title='Episode',
  yaxis_title='Total Reward',
)

fig.show()

In [10]:
# eval run

eval_env = gym.make("FrozenLake-v1", max_episode_steps=10_000, render_mode="human", is_slippery=True)

obs, info = eval_env.reset()

done = False

rewards = 0

while not done:
  eval_env.render()
  
  # choose an action
  action = select_action(q_table, obs, eval_env, 0)
  print(action)

  # take the action
  obs, reward, terminated, truncated, info = eval_env.step(action)

  rewards += float(reward)
  done = terminated or truncated

eval_env.close()
print(f"Total rewards: {rewards}")

0
0
3
0
0
3
1
1
Total rewards: 0.0


In [11]:
print(q_table)

[[0.2117461  0.16290205 0.16075855 0.16853179]
 [0.0830867  0.08231363 0.04098191 0.15317851]
 [0.11031893 0.10771368 0.11103582 0.13154774]
 [0.05322204 0.09042715 0.09168903 0.12674291]
 [0.22610439 0.11078163 0.12773209 0.13573533]
 [0.         0.         0.         0.        ]
 [0.11974431 0.06165643 0.0394764  0.03745432]
 [0.         0.         0.         0.        ]
 [0.12236764 0.13543789 0.09426274 0.25093159]
 [0.05262303 0.27158826 0.11183083 0.11788873]
 [0.13335049 0.23016694 0.11513562 0.09091393]
 [0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.05510985 0.13994825 0.02595319 0.34405913]
 [0.15666367 0.39666941 0.63833919 0.26713262]
 [0.         0.         0.         0.        ]]
